# ARC Prize 2026 — ARC-AGI-3 Online Research Run

Online research notebook driven by the ARC-AGI-3 SDK's built-in LLM agent
template. This is **not** the prize-safe Kaggle submission because official
evaluation does not allow internet/API-based systems. Use
`submission_offline.ipynb` for the offline controller.

This notebook now uses the SDK builtin LLM template
(`arc_agi_3.AVAILABLE_AGENTS["llm"]`) wired through `arc_agi_3.Swarm`. The
legacy custom Claude tool-use loop in `agent/llm_agent.py` is **parked**: it
remains in the repo for reference but is no longer wired into this notebook.

Before running:
1. Add this repo as a Kaggle Dataset named `arc-prize-2026-agent` (or uncomment the `git clone` line below).
2. Create two Kaggle Secrets: `ARC_API_KEY` and `ANTHROPIC_API_KEY` (the SDK's "llm" template reads these from env).
3. Enable Internet in the notebook settings for research runs only (required for the Anthropic and ARC APIs).

In [ ]:
# 1. Install runtime dependencies
!pip install -q arc-agi-3 arcengine anthropic

In [ ]:
# 2. Make the agent package importable.
# Option A (default): attach this repo as a Kaggle Dataset and point sys.path at it.
# Option B: clone from GitHub (uncomment after pushing).

import os, sys

DATASET_PATH = "/kaggle/input/arc-prize-2026-agent"
if os.path.isdir(DATASET_PATH):
    sys.path.insert(0, DATASET_PATH)
else:
    # Fallback: clone from GitHub.
    # !git clone https://github.com/YOUR_USERNAME/arc-prize-2026.git /kaggle/working/repo
    # sys.path.insert(0, "/kaggle/working/repo")
    raise RuntimeError(
        f"{DATASET_PATH} not found. Attach the repo as a Kaggle Dataset or uncomment the git clone lines."
    )

In [ ]:
# 3. Load secrets into env vars (both SDKs read from env).
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["ARC_API_KEY"] = secrets.get_secret("ARC_API_KEY")
os.environ["ANTHROPIC_API_KEY"] = secrets.get_secret("ANTHROPIC_API_KEY")

In [ ]:
# 4. Run the SDK's builtin "llm" agent template across every game the API exposes.
#
# Using SDK builtin LLM template (arc_agi_3.AVAILABLE_AGENTS["llm"]) wired
# through arc_agi_3.Swarm. The legacy custom Claude loop in
# `agent/llm_agent.py` is parked: still in the repo for reference, but no
# longer imported here.
import logging, json, os
import requests

import arc_agi_3
from arc_agi_3 import Swarm

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
logger = logging.getLogger("submission")

# ROOT_URL matches the SDK CLI default (arc_agi_3._cli._get_root_url):
# scheme=https, host=three.arcprize.org. Override via ARC_ROOT_URL if needed.
ROOT_URL = os.environ.get("ARC_ROOT_URL", "https://three.arcprize.org")
TAGS = ["research", "sdk-llm-template"]

# Discover games via the API (same call the SDK CLI makes).
headers = {
    "X-API-Key": os.environ.get("ARC_API_KEY", ""),
    "Accept": "application/json",
}
with requests.Session() as discovery:
    discovery.headers.update(headers)
    games_resp = discovery.get(f"{ROOT_URL}/api/games", timeout=10)
    games_resp.raise_for_status()
    games = [g["game_id"] for g in games_resp.json()]
logger.info("discovered %d games: %s", len(games), games)

assert "llm" in arc_agi_3.AVAILABLE_AGENTS, (
    "Builtin 'llm' agent template not available -- install arc-agi-3 with the openai extra."
)

swarm = Swarm(agent="llm", ROOT_URL=ROOT_URL, games=games, tags=TAGS)
scorecard = swarm.main()
payload = scorecard.model_dump() if scorecard is not None else {}
print(json.dumps(payload, indent=2, default=str))

In [ ]:
# 5. Persist the scorecard for Kaggle's grader.
out_path = "/kaggle/working/scorecard.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, default=str)
print("wrote", out_path)